# 📘 智能体架构 12：图结构 / 世界模型记忆

本篇深入探讨对 AI 智能体非常强大的一类记忆结构：**基于图的世界模型（Graph-based World Model）**。它超越简单文档检索或对话历史，构建结构化、相互关联的世界理解，更接近人类的语义记忆。

信息不是孤立文本块，而是解析为**实体（节点）**与**关系（边）**，形成可查询的丰富知识图谱。智能体可通过遍历图回答复杂问题，揭示非结构化文本中难以发现的洞见。

为详细展示，我们构建**企业情报智能体**，它将：
1. **摄入非结构化报告：** 阅读关于公司、人员与产品的文本。
2. **构建知识图谱：** 用 LLM 抽取实体（如 `Company`、`Person`）与关系（如 `ACQUIRED`、`WORKS_FOR`、`COMPETES_WITH`），写入 Neo4j 图数据库。
3. **回答复杂多跳问题：** 利用图回答需要串联多条信息的问题，例如「*收购 BetaSolutions 的那家公司里，有谁任职？*」——这类查询对标准向量检索极其困难。


### 定义

**图结构 / 世界模型记忆** 将知识存放在结构化图数据库中：节点表示实体（人、地点、概念等），边表示其间关系，形成可推理的动态「世界模型」。

### 高层工作流

1. **信息摄入：** 智能体接收非结构化或半结构化数据（文本、文档、API 响应）。
2. **知识抽取：** 由 LLM 驱动的流程解析信息，识别关键实体及其关系。
3. **图更新：** 将抽取的节点与边添加或更新到持久化图数据库（如 Neo4j）。
4. **问答 / 推理：** 面对问题时，智能体：
    a. 将自然语言问题转为形式化图查询语言（如 Neo4j 的 Cypher）。
    b. 在图上执行查询，检索相关子图或事实。
    c. 将查询结果合成为自然语言答案。

### 适用场景 / 应用
* **企业知识助理：** 从内部文档构建可查询的公司项目、员工与客户模型。
* **高级研究助理：** 通过摄入论文，动态构建某学科知识库。
* **复杂系统诊断：** 建模组件与依赖关系以定位故障。

### 优势与局限
* **优势：**
    * **结构化且可解释：** 知识组织清晰；答案可通过图上路径解释。
    * **支持复杂推理：** 擅长需要沿关系串联的多跳问题。
* **局限：**
    * **前期复杂度高：** 需要良好模式与稳健的抽取流程。
    * **图维护：** 更新、消解冲突、淘汰过时事实（知识生命周期管理）具有挑战性。


## 阶段 0：基础与环境

将安装依赖（含 Neo4j 驱动）并配置环境。**必须**有运行中的 Neo4j 实例，并在 `.env` 中配置凭据。


In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv langchain_community neo4j

In [2]:
import os
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_community.graphs import Neo4jGraph
from langchain.chains import GraphCypherQAChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel as V1BaseModel

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Graph Memory (DeepSeek)"

required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY", "NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：构建图构建智能体

这是摄入管道的核心：需要能从非结构化文本中抽取实体与关系，并以结构化格式输出。我们使用具备结构化输出能力（Pydantic）的 LLM 作为知识抽取器。


In [3]:
console = Console()
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0,
)

# Connect to our Neo4j database
try:
    graph = Neo4jGraph()
    # Clear the graph for a clean run
    graph.query("MATCH (n) DETACH DELETE n")
except Exception as e:
    console.print(f"[bold red]Failed to connect to Neo4j: {e}. Please check your credentials and connection.[/bold red]")
    graph = None

# Pydantic models for structured extraction (using LangChain's v1 BaseModel for compatibility with older structured output methods)
class Node(V1BaseModel):
    id: str = Field(description="Unique name or identifier for the entity.")
    type: str = Field(description="The type or label of the entity (e.g., Person, Company, Product).")

class Relationship(V1BaseModel):
    source: Node
    target: Node
    type: str = Field(description="The type of relationship (e.g., WORKS_FOR, ACQUIRED).")

class KnowledgeGraph(V1BaseModel):
    """A graph of nodes and relationships."""
    relationships: List[Relationship]

# The Graph Maker Agent
def get_graph_maker_chain():
    extractor_llm = llm.with_structured_output(KnowledgeGraph)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert at extracting information from text and building a knowledge graph. Extract all entities (nodes) and relationships from the provided text. The relationship type should be a verb in all caps, like 'WORKS_FOR' or 'ACQUIRED'."),
        ("human", "Extract a knowledge graph from the following text:\n\n{text}")
    ])
    return prompt | extractor_llm

graph_maker_agent = get_graph_maker_chain()
print("Successfully connected to Neo4j and defined the Graph Maker Agent.")

Successfully connected to Neo4j and defined the Graph Maker Agent.


## 阶段 2：摄入知识并构建世界模型

向智能体喂入一系列相关但独立的文档；逐条处理并逐步构建企业知识图谱。这模拟真实系统随新信息到达持续学习的过程。


In [4]:
unstructured_documents = [
    "On May 15, 2023, global tech giant AlphaCorp announced its acquisition of startup BetaSolutions, a leader in cloud-native database technology.",
    "Dr. Evelyn Reed, a renowned AI researcher, has been the Chief Science Officer at AlphaCorp since 2021. She leads the team responsible for the QuantumLeap AI platform.",
    "Innovate Inc.'s flagship product, NeuraGen, is seen as a direct competitor to AlphaCorp's QuantumLeap AI. Meanwhile, Innovate Inc. recently hired Johnathan Miles as their new CTO."
]
for i, doc in enumerate(unstructured_documents):
    console.print(f"--- Ingesting Document {i+1} ---")
    try:
        kg_data = graph_maker_agent.invoke({"text": doc})
        if kg_data.relationships:
            graph.add_graph_documents(graph_documents=kg_data.relationships, include_source=False)
            console.print(f"[green]Successfully added {len(kg_data.relationships)} relationships to the graph.[/green]")
        else:
             console.print("[yellow]No relationships extracted.[/yellow]")
    except Exception as e:
        console.print(f"[red]Failed to process document: {e}[/red]")

console.print("--- ✅ Knowledge Graph Ingestion Complete ---")
console.print("\n--- Graph Schema ---")
console.print(graph.schema)

--- Ingesting Document 1 ---
Successfully added 1 relationships to the graph.
--- Ingesting Document 2 ---
Successfully added 2 relationships to the graph.
--- Ingesting Document 3 ---
Successfully added 2 relationships to the graph.
--- ✅ Knowledge Graph Ingestion Complete ---



--- Graph Schema ---


Node properties: [{'properties': [('id', 'STRING')], 'labels': ['Product']}, {'properties': [('id', 'STRING')], 'labels': ['Person']}, {'properties': [('id', 'STRING')], 'labels': ['Company']}]
Relationship properties: []
Relationships: [(:Company)-[:PRODUCES]->(:Product), (:Person)-[:WORKS_FOR]->(:Company), (:Product)-[:COMPETES_WITH]->(:Product), (:Company)-[:ACQUIRED]->(:Company)]


## 阶段 3：构建图查询智能体

知识图谱就绪后，需要能使用它的智能体。核心是 **文本到 Cypher（Text-to-Cypher）** 流程：接收自然语言问题，在图模式上下文中转为 Cypher，执行查询，再将结果合成为人类可读答案。


In [5]:
# LangChain has a built-in chain for this, but we'll inspect the components
# to understand how it works.
cypher_generation_prompt = ChatPromptTemplate.from_template(
    """You are an expert Neo4j Cypher query developer. Your task is to convert a user's natural language question into a valid Cypher query.
You must use the provided graph schema to construct the query. Do not use any node labels or relationship types that are not in the schema.
Return ONLY the Cypher query, with no additional text or explanations.

Graph Schema:
{schema}

User Question:
{question}
"""
)

cypher_response_prompt = ChatPromptTemplate.from_template(
    """You are an assistant that provides clear, natural language answers based on query results from a knowledge graph.
Use the context from the graph query result to answer the user's original question.

User Question: {question}
Query Result: {context}
"""
)

def query_graph(question: str) -> Dict[str, Any]:
    """The full Text-to-Cypher and synthesis pipeline."""
    console.print(f"\n[bold]Question:[/bold] {question}")
    
    # 1. Generate Cypher Query
    console.print("--- ➡️ Generating Cypher Query ---")
    cypher_chain = cypher_generation_prompt | llm
    generated_cypher = cypher_chain.invoke({"schema": graph.schema, "question": question}).content
    console.print(f"[cyan]Generated Cypher:\n{generated_cypher}[/cyan]")
    
    # 2. Execute Cypher Query
    console.print("--- ⚡ Executing Query ---")
    try:
        context = graph.query(generated_cypher)
        console.print(f"[yellow]Query Result:\n{context}[/yellow]")
    except Exception as e:
        console.print(f"[red]Cypher Query Failed: {e}[/red]")
        return {"answer": "I was unable to execute a query to find the answer to your question."}
    
    # 3. Synthesize Final Answer
    console.print("--- 🗣️ Synthesizing Final Answer ---")
    synthesis_chain = cypher_response_prompt | llm
    answer = synthesis_chain.invoke({"question": question, "context": context}).content
    
    return {"answer": answer}

print("Graph-Querying Agent defined successfully.")

Graph-Querying Agent defined successfully.


## 阶段 4：演示与分析

进行最终测试：问题从简单事实检索到需要串联三份来源文档的复杂多跳推理。


In [6]:
# Test 1: Simple fact retrieval (requires info from doc 2)
result1 = query_graph("Who works for AlphaCorp?")
console.print("\n--- Final Answer ---")
console.print(Markdown(result1['answer']))

# Test 2: Another simple fact retrieval (requires info from doc 1)
result2 = query_graph("What company did AlphaCorp acquire?")
console.print("\n--- Final Answer ---")
console.print(Markdown(result2['answer']))

# Test 3: The multi-hop reasoning question (requires info from all 3 docs)
result3 = query_graph("What companies compete with the products made by the company that acquired BetaSolutions?")
console.print("\n--- Final Answer ---")
console.print(Markdown(result3['answer']))


Question: Who works for AlphaCorp?
--- ➡️ Generating Cypher Query ---
Generated Cypher:
MATCH (p:Person)-[:WORKS_FOR]->(c:Company {id: 'AlphaCorp'}) RETURN p.id
--- ⚡ Executing Query ---
Query Result:
[{'p.id': 'Dr. Evelyn Reed'}]
--- 🗣️ Synthesizing Final Answer ---



--- Final Answer ---


Dr. Evelyn Reed works for AlphaCorp.


Question: What company did AlphaCorp acquire?
--- ➡️ Generating Cypher Query ---
Generated Cypher:
MATCH (:Company {id: 'AlphaCorp'})-[:ACQUIRED]->(acquired_company:Company)
RETURN acquired_company.id
--- ⚡ Executing Query ---
Query Result:
[{'acquired_company.id': 'BetaSolutions'}]
--- 🗣️ Synthesizing Final Answer ---



--- Final Answer ---


AlphaCorp acquired BetaSolutions.


Question: What companies compete with the products made by the company that acquired BetaSolutions?
--- ➡️ Generating Cypher Query ---
Generated Cypher:
MATCH (acquirer:Company)-[:ACQUIRED]->(:Company {id: 'BetaSolutions'})
MATCH (acquirer)-[:PRODUCES]->(product:Product)
MATCH (product)-[:COMPETES_WITH]->(competitor_product:Product)
MATCH (competitor_company:Company)-[:PRODUCES]->(competitor_product)
RETURN DISTINCT competitor_company.id
--- ⚡ Executing Query ---
Query Result:
[{'competitor_company.id': 'Innovate Inc.'}]
--- 🗣️ Synthesizing Final Answer ---



--- Final Answer ---


Innovate Inc. competes with the products made by the company that acquired BetaSolutions.

### 结果分析

演示凸显了基于图的世界模型的深层优势：

- 前两个问题是简单查找：智能体成功将问题转为 Cypher、查询图并找到直接关系。

- 第三个问题至关重要：标准 RAG 智能体往往会失败——它可能分别找到收购文档与竞争对手文档，但难以把文档 1 中的「AlphaCorp」与文档 2、3 中的同一实体关联起来，缺少显式关系结构。

- 而我们的基于图的智能体轻松完成。可从生成的 Cypher 直接追踪逻辑：
    1. `MATCH (acquirer:Company)-[:ACQUIRED]->(:Company {id: 'BetaSolutions'})`：先找到收购 BetaSolutions 的公司（结果：AlphaCorp）。
    2. `MATCH (acquirer)-[:PRODUCES]->(product:Product)`：再找该公司产品（结果：QuantumLeap AI）。
    3. `MATCH (product)-[:COMPETES_WITH]->(competitor_product:Product)`：再找与之竞争的产品（结果：NeuraGen）。
    4. `MATCH (competitor_company:Company)-[:PRODUCES]->(competitor_product)`：最后找生产竞争产品的公司（结果：Innovate Inc.）。

沿关系遍历并综合不同来源的信息，是图结构 / 世界模型架构的「超能力」。答案不仅是检索，更包含推理。


## 小结

本 Notebook 围绕 **图结构 / 世界模型记忆** 构建了完整智能体系统，演示了全生命周期：摄入非结构化数据、用 LLM 构建结构化知识图谱，再用图回答需要真正推理的复杂多跳问题。

相比更简单的记忆系统，该架构能力跃升显著。通过建立显式、可查询的世界模型，智能体能够连接分散事实并发现隐含洞见。尽管长期维护图仍有挑战，但构建深度知识型、可解释助理的潜力使其成为现代智能体设计中最令人兴奋、最强大的模式之一。
